# Day 6 — Notebook 04b: Data Leakage in Simple Words

**Data leakage = the model accidentally sees the answers before the exam.**
It scores brilliantly in testing… then fails in the real world.

We'll look at 2 tiny examples you can read completely:
1. **Using a clue from the future** (target leakage)
2. **Peeking at test data when calculating an average** (preprocessing leakage)

**Action:** Click **Run → Run All Cells**.

## Example 1: Using a Clue from the Future

Below are 8 past StreamPulse customers.

`cancellation_reason` only gets filled in **after** someone cancels — it's a clue from the future!

In [5]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Past customers (we know what happened)
train = pd.DataFrame({
    "logins_per_month":    [25, 2, 20, 1, 18, 3, 22, 4],
    "cancellation_reason": [ 0, 1,  0, 1,  0, 1,  0, 1],   # 1 = reason filled in (only AFTER cancelling!)
    "churned":             [ 0, 1,  0, 1,  0, 1,  0, 1],
})
train

,logins_per_month,cancellation_reason,churned
0,25,0,0
1,2,1,1
2,20,0,0
3,1,1,1
4,18,0,0
5,3,1,1
6,22,0,0
7,4,1,1


Now next month's customers arrive. **Nobody has cancelled yet**, so `cancellation_reason` is `0` for everyone.

In [6]:
new_customers = pd.DataFrame({
    "logins_per_month":    [2, 24, 1],
    "cancellation_reason": [0,  0, 0],
})
actual = [1, 0, 1]   # what really happened later
new_customers

,logins_per_month,cancellation_reason
0,2,0
1,24,0
2,1,0


### ❌ Leaky model — learns from the "future" column

In [7]:
leaky = DecisionTreeClassifier(random_state=0).fit(train[["cancellation_reason"]], train["churned"])

print("Leaky training accuracy:", leaky.score(train[["cancellation_reason"]], train["churned"]), "  <- 100%! looks perfect")
print("Leaky predictions      :", leaky.predict(new_customers[["cancellation_reason"]]).tolist(), "  <- says NOBODY will cancel")
print("What really happened   :", actual)

Leaky training accuracy: 1.0   <- 100%! looks perfect
Leaky predictions      : [0, 0, 0]   <- says NOBODY will cancel
What really happened   : [1, 0, 1]


### ✅ Honest model — uses only what we know BEFORE they cancel

In [8]:
honest = DecisionTreeClassifier(random_state=0).fit(train[["logins_per_month"]], train["churned"])

print("Honest predictions     :", honest.predict(new_customers[["logins_per_month"]]).tolist(), "  <- correct!")
print("What really happened   :", actual)

Honest predictions     : [1, 0, 1]   <- correct!
What really happened   : [1, 0, 1]


**What happened?**
- The leaky model learned *"cancellation_reason = 1 → churn"* → perfect score in training.
- But for new customers that column is always `0` (nobody has cancelled yet), so it predicts **no one** will churn.
- The honest model used **logins** (known in advance) and got all 3 right.

👉 **Rule 1:** Ask *"Would I know this value BEFORE the event happens?"* If not, don't use it.

## Example 2: Peeking at Test Data When Calculating an Average

We want to fill missing ages with the **average age**.

| | Ages |
|---|---|
| Train customers | 20, 30, 40 |
| Test customers (the "exam") | 90 |

In [ ]:
train_ages = pd.Series([20, 30, 40])
test_ages  = pd.Series([90])

# ❌ WRONG: average of ALL data (includes the test customer)
leaky_avg = pd.concat([train_ages, test_ages]).mean()

# ✅ RIGHT: average of TRAIN data only
clean_avg = train_ages.mean()

print("Leaky average:", leaky_avg, "  <- the test customer (age 90) pulled it up = peeked!")
print("Clean average:", clean_avg, "  <- honest")

**What happened?**
The test customer is supposed to be *unknown*. By including them in the average, training got a hint about the exam.

👉 **Rule 2:** **Split first, then calculate.** Averages, scaling, and filling blanks must be learned from **training data only**.

In real projects, an sklearn `Pipeline` does this for you automatically:
```python
model = Pipeline([("impute", SimpleImputer(strategy="mean")), ("clf", LogisticRegression())])
model.fit(X_train, y_train)   # learns averages from TRAIN only
```

## Remember 🧠

| Mistake | Fix |
|---|---|
| Using a column from the future | Only use info known **before** the prediction |
| Calculating averages / scaling on all data | **Split first**, learn from train only |
| Score looks too good to be true (e.g. 100%) | Suspect leakage and check your features! |

> **The test set is a sealed envelope — don't open it until the very end.**